# VoiceLK — speaker_01 VITS training on Kaggle

Trains a single-speaker VITS model on the speaker_01 recordings with
`model_training/train_speaker.py`.

**Before running**

1. Upload the recordings as a Kaggle Dataset (e.g. `voicelk-speaker-01`) with the wav files
   named after the CSV's `file_name` column (`sin_s1_0001.wav`, ...), in any folder of the
   dataset. Any sample rate or channel count works; they are converted to 22050 Hz mono.
2. *Add Input* → that dataset. For a resumed run, also add the previous run's notebook
   output (see section 5).
3. *Settings* → Accelerator **GPU T4 x2**, Internet **on**.
4. If the GitHub repository is private, add a secret named `GITHUB_TOKEN`
   (*Add-ons* → *Secrets*) holding a read-only personal access token.
5. Run unattended with *Save Version* → **Save & Run All (Commit)**. An interactive session
   stops when the browser goes idle; a committed run keeps going for up to 12 hours and
   keeps everything written under `/kaggle/working`.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/RootCode-AI/voicelk_ml.git"
BRANCH = "Tharinda/newDev_V1.0"

SPEAKER = "speaker_01"
# Transcript CSV: None uses data/prepared/<SPEAKER>.csv from the repository; set a path
# under /kaggle/input to use a CSV uploaded with the dataset instead
CSV_PATH = None
SAMPLE_RATE = 22050

# Resume: a folder under /kaggle/input holding an earlier output of this notebook
# (the newest run folder inside it is picked). Leave empty for a fresh run.
RESUME_FROM = ""

EPOCHS = 1000
BATCH_SIZE = 16

WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/voicelk_ml"
DATA_DIR = f"{WORK}/data"
RUNS_DIR = f"{WORK}/runs/{SPEAKER}"

## 2. Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, sys
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 3. Code — clone the repository and switch to `BRANCH`

Clones on the first run; on a re-run it fetches and switches to `BRANCH`, so a changed
`BRANCH` or newly pushed commits are always picked up.

In [ ]:
import os, subprocess

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret; cloning anonymously (works for a public repository)")
auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL


def git(*args):
    # Commands are not echoed, so the token in auth_url never reaches the saved output
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.replace(token, "***") if token else result.stderr)
    return result.stdout.strip()


# 1. Clone (only the first time)
if not os.path.exists(f"{REPO_DIR}/.git"):
    print(f"Cloning {REPO_URL} ...")
    git("clone", auth_url, REPO_DIR)

# 2. Fetch the branch, switch to it and bring it up to date with the remote
git("-C", REPO_DIR, "fetch", auth_url, f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}")
git("-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}")
git("-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}")

print("Branch:", git("-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"))
print("Commit:", git("-C", REPO_DIR, "log", "--oneline", "-1"))

In [ ]:
# torch/torchaudio come with the Kaggle image; this installs the rest of the training
# dependencies (the TTS and trainer packages themselves are vendored in the repository)
!pip install -q -r {REPO_DIR}/model_training/requirements.txt

## 4. Data

In [ ]:
import csv, glob

csv_path = CSV_PATH or f"{REPO_DIR}/data/prepared/{SPEAKER}.csv"
with open(csv_path, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(f"{len(rows)} transcript rows in {csv_path}")

# Find the dataset folder that holds the recordings by looking for the first file name
first = rows[0]["file_name"].strip()
first = first if first.endswith(".wav") else first + ".wav"
hits = glob.glob(f"/kaggle/input/**/{first}", recursive=True)
assert hits, f"{first} not found under /kaggle/input - is the recordings dataset added as input?"
WAV_DIR = os.path.dirname(hits[0])
print("Recordings:", WAV_DIR, "-", len(glob.glob(f"{WAV_DIR}/*.wav")), "wav files")

In [ ]:
METADATA = f"{SPEAKER}_metadata.txt"
!cd {REPO_DIR} && python data_preparation/prepare_speaker_dataset.py \
    --csv "{csv_path}" --wav-dir "{WAV_DIR}" --wav-out "{DATA_DIR}/wavs" \
    --sample-rate {SAMPLE_RATE} --out "{DATA_DIR}/{METADATA}"

In [ ]:
# Phoneme coverage of the transcript CSV: rare phonemes are the ones to record more of
!cd {REPO_DIR} && python data_preparation/phoneme_coverage.py --csv "{csv_path}" | head -60

## 5. Resume (optional)

Kaggle stops a session after 12 hours. To continue: open this notebook's *Output* tab from
the finished version, *New Dataset* from it (or add the notebook output directly via
*Add Input* → *Your Work*), add it as input, set `RESUME_FROM` to its folder and commit again.
The run folder is copied into `/kaggle/working` because inputs are read-only.

In [ ]:
import shutil

continue_args = ""
if RESUME_FROM:
    runs = sorted(
        (os.path.dirname(p) for p in glob.glob(f"{RESUME_FROM}/**/config.json", recursive=True)),
        key=lambda d: max(os.path.getmtime(os.path.join(d, f)) for f in os.listdir(d)),
    )
    assert runs, f"No run folder with config.json under {RESUME_FROM}"
    target = f"{RUNS_DIR}/{os.path.basename(runs[-1])}"
    if not os.path.exists(target):
        shutil.copytree(runs[-1], target)
    continue_args = f'--continue_path "{target}"'
    print("Resuming", target)
else:
    print("Fresh run")

## 6. Train

In [ ]:
!cd {REPO_DIR} && python model_training/train_speaker.py \
    --data-path "{DATA_DIR}" --metadata {METADATA} \
    --run-name voicelk_vits_{SPEAKER} --output-path "{RUNS_DIR}" \
    --epochs {EPOCHS} --batch-size {BATCH_SIZE} {continue_args}

## 7. Output

The run folder under `/kaggle/working/runs` holds `config.json`, `best_model.pth`, the last
two `checkpoint_*.pth` and the TensorBoard logs. The converted audio is removed so the saved
output stays small - it is rebuilt from the dataset on every run.

In [ ]:
shutil.rmtree(f"{DATA_DIR}/wavs", ignore_errors=True)
for run in glob.glob(f"{RUNS_DIR}/*/"):
    print(run)
    for f in sorted(os.listdir(run)):
        print(f"  {f:45s} {os.path.getsize(os.path.join(run, f)) / 1e6:9.1f} MB")